In [ ]:
# parser 1
from scapy.all import rdpcap, UDP
import pynmea2
import json

def load_pcap(filepath):
    """Extract raw NMEA sentences from a pcap file."""
    packets = rdpcap(filepath)
    sentences = []

    for packet in packets:
        if UDP in packet:
            try:
                payload = bytes(packet[UDP].payload).decode("ascii").strip()
                if payload.startswith("$"):
                    sentences.append(payload)
            except UnicodeDecodeError:
                pass

    return sentences


def parse_sentence(raw):
    try:
        return pynmea2.parse(raw)
    except pynmea2.ParseError as e:
        print(f"[WARN] Could not parse: {raw!r} — {e}")
        return None


def assess_quality(rmc):
    if rmc.status == "V":
        return 0.0
    score = 1.0
    hdop = getattr(rmc, "horizontal_dil", None)
    if hdop:
        hdop = float(hdop)
        if hdop > 2.0:
            score -= 0.3
        elif hdop > 1.0:
            score -= 0.1
    return round(score, 2)


def build_track(rmc, hdt=None):
    confidence = assess_quality(rmc)
    return {
        "trackId":   f"TRK-{rmc.timestamp}",
        "type":      "SURFACE",
        "source":    "NMEA_GNSS",
        "timestamp": rmc.datetime.isoformat() if rmc.datetime else None,
        "position": {
            "lat": rmc.latitude  if rmc.lat  else None,
            "lon": rmc.longitude if rmc.lon  else None,
        },
        "velocity": {
            "speedKnots":       float(rmc.spd_over_grnd) if rmc.spd_over_grnd else None,
            "courseOverGround": float(rmc.true_course)   if rmc.true_course   else None,
            "trueHeading":      float(hdt.heading)       if hdt               else None,
        },
        "pntConfidence": confidence,
        "gnssStatus":    rmc.status,
    }


def process_pcap(filepath):
    sentences = load_pcap(filepath)
    print(f"✓ Extracted {len(sentences)} NMEA sentences from pcap\n")

    tracks = []
    last_hdt = None

    for raw in sentences:
        msg = parse_sentence(raw)
        if msg is None:
            continue

        if isinstance(msg, pynmea2.HDT):
            last_hdt = msg

        elif isinstance(msg, pynmea2.RMC):
            track = build_track(msg, hdt=last_hdt)
            tracks.append(track)
            print(json.dumps(track, indent=2, default=str))

    return tracks


if __name__ == "__main__":
    results = process_pcap("kraken_data/multicast_AdvNavData-2026-05-08_07-40-14_capture.pcap")
    print(f"\n✓ Processed {len(results)} track updates")

In [3]:
# parser two with file handling
from scapy.all import PcapReader, UDP
import pynmea2
import json

def load_pcap(filepath, max_sentences=1000):
    """Extract raw NMEA sentences from a pcap file, streaming packet by packet."""
    sentences = []
    with PcapReader(filepath) as reader:
        for packet in reader:
            if len(sentences) >= max_sentences:
                break
            if UDP in packet:
                try:
                    payload = bytes(packet[UDP].payload).decode("ascii").strip().rstrip("\x00")
                    if payload.startswith("$"):
                        sentences.append(payload)
                except UnicodeDecodeError:
                    pass
    return sentences

def parse_sentence(raw):
    try:
        return pynmea2.parse(raw)
    except pynmea2.ParseError as e:
        print(f"[WARN] Could not parse: {raw!r} — {e}\n")
        return None

def assess_quality(rmc):
    if rmc.status == "V":
        return 0.0
    score = 1.0
    hdop = getattr(rmc, "horizontal_dil", None)
    if hdop:
        hdop = float(hdop)
        if hdop > 2.0:
            score -= 0.3
        elif hdop > 1.0:
            score -= 0.1
    return round(score, 2)

def build_track(rmc, hdt=None):
    confidence = assess_quality(rmc)
    return {
        "trackId":       f"TRK-{rmc.timestamp}",
        "sentenceType":  "RMC+HDT" if hdt else "RMC",
        "type":          "SURFACE",
        "source":        "NMEA_GNSS",
        "timestamp":     rmc.datetime.isoformat() if rmc.datetime else None,
        "position": {
            "lat": rmc.latitude  if rmc.lat  else None,
            "lon": rmc.longitude if rmc.lon  else None,
        },
        "velocity": {
            "speedKnots":       float(rmc.spd_over_grnd) if rmc.spd_over_grnd else None,
            "courseOverGround": float(rmc.true_course)   if rmc.true_course   else None,
            "trueHeading":      float(hdt.heading)       if hdt and hdt.heading else None,
        },
        "pntConfidence": confidence,
        "gnssStatus":    rmc.status,
    }

def process_pcap(filepath, output_file="tracks_output.json", max_sentences=1000):
    sentences = load_pcap(filepath, max_sentences=max_sentences)
    print(f"✓ Extracted {len(sentences)} NMEA sentences (capped at {max_sentences})\n")
    tracks = []
    last_hdt = None
    for raw in sentences:
        msg = parse_sentence(raw)
        if msg is None:
            continue
        if isinstance(msg, pynmea2.HDT):
            last_hdt = msg
        elif isinstance(msg, pynmea2.RMC):
            track = build_track(msg, hdt=last_hdt)
            tracks.append(track)
    with open(output_file, "w") as f:
        json.dump(tracks, f, indent=2, default=str)
    print(f"✓ Processed {len(tracks)} track updates")
    print(f"✓ Results written to {output_file}")
    return tracks

if __name__ == "__main__":
    results = process_pcap("kraken_data/multicast_AdvNavData-2026-05-08_07-40-14_capture.pcap")
    print(f"\n✓ Processed {len(results)} track updates")

✓ Extracted 1000 NMEA sentences (capped at 1000)

✓ Processed 284 track updates
✓ Results written to tracks_output.json

✓ Processed 284 track updates


In [ ]:
from scapy.all import rdpcap, UDP

def inspect_pcap(filepath):
    packets = rdpcap(filepath)
    sentence_types = {}

    for packet in packets:
        if UDP in packet:
            try:
                payload = bytes(packet[UDP].payload).decode("ascii").strip()
                if payload.startswith("$"):
                    # Extract just the sentence type e.g. GPRMC, GPHDT
                    talker_and_type = payload[1:].split(",")[0]
                    sentence_types[talker_and_type] = sentence_types.get(talker_and_type, 0) + 1
            except UnicodeDecodeError:
                pass

    print("Sentence types found:")
    for k, v in sorted(sentence_types.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v} packets")

inspect_pcap("kraken_data/multicast_AdvNavData-2026-05-08_07-40-14_capture.pcap")